# 02 · Draw your sample, and put it in front of your coders

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/egumasa/lda2-final-template/blob/main/notebooks/02_sample.ipynb)

Choose which items your study is about — and be able to say why those.

```
  01_build_pool_<track>  →▶ 02_sample  →  02b_add_samples  →  03_annotate  →  04_develop  →  05_test  →  06_report
```

| | |
|---|---|
| **Reads** | `data/pools/<track>_pool.json` (from 01) |
| **Writes** | `data/gold/<track>_<group>_sample.json`, and a Google Sheet to annotate |

---

**One sitting, start to finish.** You pick a sampling strategy, draw the sample, save it, and create the annotation sheet — then the notebook is done, and the week's real work happens in the sheet.

Notebook 03 is where you come back to, once both coders have finished. Keeping that separate is deliberate: re-running the cells below *after* annotating would redraw the sample your sheet was built from.

> Budget real time for the annotation itself. Forty items, two annotators, plus the argument afterwards. It is the most valuable thing you will make this week and the easiest to rush.

### Working as a group

These notebooks live in a shared Drive folder, so **all of you can edit at once** — Colab syncs edits like a Google Doc. Two things do *not* work that way:

- **Runtimes are per-person.** Seeing `sampled` in a saved output does not mean `sampled` exists in *your* session. Whoever runs the cells is the **driver**.
- **Files are last-write-wins.** `data/`, `prompts/` and `outputs/` are ordinary files, not Google Docs. Two of you writing the same one does not merge them — Drive keeps one and may quietly leave the other beside it as `… (1).json`, which nothing downstream will ever read. Let the driver be the only one running cells that write.

The **annotation Sheet is the exception** — that is a real Google Sheet, so annotate it together, all at once.

## Setup — run this first

This cell mounts your Google Drive and finds your group's shared folder, `lda2-final-template`. Everything the project produces — the pool, the gold set, your prompts, the outputs — is an ordinary file in there, which is what makes it survive the runtime resetting *and* lets the rest of your group see it.

**One member sets the folder up once:**

1. That member runs the `git clone` line this cell prints if the folder is missing, which puts it in their own Drive.
2. They share it with the group (right-click ▸ *Share*), with edit access.
3. Everyone else opens *Shared with me*, right-clicks the folder, and chooses **Add shortcut to Drive** ▸ *My Drive*.

Keep that shortcut's name exactly `lda2-final-template`. It is what makes the same path work for all of you — if Drive renames it to `lda2-final-template (1)`, this cell will not find it.

From then on, open notebooks from the folder itself (*File ▸ Open notebook ▸ Drive*) rather than from the GitHub badge, so you are working on your group's copy and not a fresh one.

**Looking inside a helper.** The functions this cell imports are defined in `scripts/`. Two ways to read one, both the same ones you used on Day 2:

- `help(save_json)` prints its first line — what to pass in and what comes back — and the description of each argument. Typing `save_json(` and pressing **Shift+Tab** shows the same thing in a pop-up.
- To read the code itself, open `scripts/pipeline.py` from the **Files** panel on the left. Colab lists the functions in that file down the side, so you can click straight to the one you want.

In [ ]:
# ------------------------------------------------------------------
# SETUP — run me first. You are not expected to read it.
# ------------------------------------------------------------------
# This cell is plumbing, and it is the only cell in the project that is.
# It finds your group's shared folder in Google Drive, because everything
# this project keeps goes in there: a Colab runtime is wiped when it resets,
# and nobody else in your group can see inside it. Then it makes the
# project's own code importable. Run it and move on; nothing below asks you
# to have understood it.

FOLDER = "lda2-final-template"     # the shared folder, in every member's Drive

import os, sys

PROJECT = ".."                              # running locally: it is just above us

try:
    from google.colab import drive           # only exists inside Colab
except ImportError:
    pass
else:
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/" + FOLDER
    if not os.path.isdir(PROJECT):
        raise RuntimeError(
            "Could not find " + PROJECT + "\n\n"
            "Setting the folder up for your group? Run this in a new cell:\n"
            "  !git clone https://github.com/egumasa/lda2-final-template.git "
            + PROJECT + "\n"
            "then share the folder with the rest of your group.\n\n"
            "Someone else already did? Open Drive, find the folder under "
            "'Shared with me', right-click it, and choose 'Add shortcut to "
            "Drive'. Keep the name exactly " + FOLDER + ".")
    # Work inside the project folder, where the notebooks live.
    os.makedirs(PROJECT + "/notebooks", exist_ok=True)
    os.chdir(PROJECT + "/notebooks")

# scripts/ and config.py, by their real paths - so they are found from wherever
# this notebook happens to be working.
sys.path.append(PROJECT)
sys.path.append(PROJECT + "/scripts")

# Re-read config.yaml every time this cell runs. Without the reload, Python
# hands back the settings it read the FIRST time, and editing config.yaml
# would appear to do nothing until you restarted the runtime.
import importlib
import config
importlib.reload(config)

# Named one by one rather than with `import *`, so that every name a cell
# below uses can be traced back to the file it came from — config.yaml for
# these, scripts/ for the rest.
from config import (TRACK, GROUP, RUN, SEED, N_PER_CLASS, DEV, CODERS,
                    MEMBERS, LABELS_ORDER, TEMPERATURE, MODEL, ROOT, OUT_DIR,
                    POOL_PATH, DEMO_POOL_PATH, SAMPLE_PATH, GOLD_PATH,
                    SAMPLE_BEFORE_TOPUP_PATH, DEV_PATH, TEST_PATH,
                    DISAGREED_PATH, ADJUDICATED_PATH, PRED_PATH, ROUNDS_PATH,
                    NOTES_PATH, TESTLOG_PATH, PROMPT_FILE, SHEET_PATH,
                    describe)

# Reading and writing files, and making the Google Sheet, are plumbing, so
# they are imported. So is the drawing itself: sample_pool, sample_random and
# sample_by_document are three ways of picking items out of a list, and there
# is no judgment inside any of them. The judgment is WHICH of the three, and
# that is the one thing not imported — `sample` is defined further down, in a
# cell you can read and change, just before the step that calls it.
from pipeline import (load_gold, save_json, label_set,
                      sample_pool, sample_random, sample_by_document)
from annotate import create_annotation_sheet

describe()                  # what this notebook is working on


> **Everything above comes from `config.yaml`** — one small file at the top of the repo, which you edit once as a group, and the only file in the plumbing you touch. That is deliberate: the seed that drew your sample has to be the seed you report, and five copies of a number in five notebooks is five chances for them to disagree. Your settings are also the filenames — `track: cars50`, `group: kimura`, `run: v1` means this notebook reads and writes `cars50_kimura_v1_...`. If the line it just printed is not your track, your group and your seed, fix `config.yaml` and re-run this cell.

## What this notebook is for

A **pool** is everything the corpus has, with its natural imbalance. A **sample** is the balanced subset you actually study — equal items per label, so precision, recall, F1 and the confusion matrix all stay readable. Rare labels simply yield fewer; that is a property of the data, and it belongs in your limitations.

Keeping the two separate is also what leaves unused items available as few-shot examples in notebook 04, without showing the model the answers you are testing it on. That is why you sample rather than just taking the first 40 rows.

> **Do not have a pool yet?** Run `01_build_pool_<track>.ipynb` first. To see this notebook work before then, point at `DEMO_POOL_PATH` in the cell below — but the demo pools are small enough that a real sample would eat most of them, and `sample_pool` will warn you when it does.

First we open the pool file and count what is in it. `load_gold` is the same call you ran on Day 2 S5 step F and again at the start of Day 3; it lives in `scripts/pipeline.py`. The only decision here is which of the two paths to open.

In [ ]:
# ══ STEP 1 · Load the pool ════════════════════════════════════════════════
# Opens the pool file notebook 01 wrote and counts the items under each label.
# Creates: pool

pool = load_gold(POOL_PATH)     # or DEMO_POOL_PATH to try it out first

from collections import Counter
print(len(pool), "items")
print(Counter(item["label"] for item in pool))


## The decision this notebook turns on: **how do you draw the sample?**

Forty items out of a few thousand. *Which* forty is not a technicality — it decides what your F1 is a statement about, and it is the first thing a reader of your report is entitled to ask. Three defensible answers, and they disagree with each other:

| Strategy | What you get | What you can claim | What it costs |
|---|---|---|---|
| **Balanced by label** | equal items per label | clean per-class precision and recall; a readable confusion matrix | your sample no longer looks like the corpus — you have over-sampled the rare moves on purpose |
| **Simple random** | the corpus as it is | "this is how the model does on text like this" | a rare label may arrive with two items, or none, and F1 on a class of two means very little |
| **By document** | whole passages, sampled | "forty sentences from forty abstracts" rather than from three | labels are not controlled at all; only `cars50` and `raamove` can do it |

### The one that is easy to get wrong

Forty sentences drawn from three introductions and forty drawn from forty are both "n = 40", and they support very different claims. Three long introductions is *three* texts, three authors, three topics — and neighbouring sentences in one passage are not independent observations of anything. If you sample balanced or at random on `cars50` or `raamove`, print how many distinct `doc_id`s you ended up with before you write "we sampled 40 sentences".

### And the ceiling on size

Whatever you choose, the smallest class is a hard ceiling on a balanced draw: ask for more than a class has and you get everything it has, and your sample is quietly no longer balanced. The other ceiling is time — every item is one API call in notebook 04, times the number of rounds, and one row two of you must annotate by hand below. **Around 40 items total** is the size this project is built for. Set `n_per_class` in `config.yaml` and re-run the setup cell.

Whichever you pick: say so in `PLAN.md`, and do not change it after you have seen the numbers.

### The code that draws the sample

This is your **sampling method** — what your report's methodology section has to describe and the Q&A may well ask you to defend, so it is in front of you rather than behind an import. SETUP did not import these four: **the cells below are where they come from**, so run them before the step that uses them. They are read out of `scripts/pipeline.py` when this notebook is generated, so they are not a simplified copy — they are the code that runs.

They are ordinary definitions, which means you can change one. Edit a cell, run it again, and the draw below behaves differently. (To get the original back, re-run the notebook from the top.)

**Where the reproducibility comes from**, in all three strategies: `random.Random(seed)` — one generator, made from your seed, doing every shuffle. Same seed, same draw, on anyone's machine. That one line is the whole of why your sample is checkable by someone else.

First, the helpers the definitions below call. Nothing to decide here — run it and read on.

In [ ]:
import random
from pipeline import _report_draw, label_set, reid

**`sample`** is the one you call, and the one word you change is its `strategy`. In: the pool, a strategy name, `n_per_class`, the seed. Out: the list of sampled items. It sizes all three strategies from the same `n_per_class`, so switching changes *how* the items are chosen and not *how many* — which is what makes their counts comparable. It does no drawing itself; it hands the work to one of the next three.

In [ ]:
def sample(pool: list[dict[str, str]],
           strategy: str,
           n_per_class: int,
           seed: int = 42,
           n_per_doc: int = 4) -> list[dict[str, str]]:
    """Draw a sample using one of the three strategies, all sized from n_per_class.

    Args:
        pool: the items to draw from.
        strategy: which draw to make.
            "balanced" - up to n_per_class items of EACH label.
            "random" - the same TOTAL, drawn without regard to label.
            "by_document" - whole passages, enough to reach that total. cars50 and
            raamove only; the other tracks have no documents.
        n_per_class: how many items per label. Every strategy is sized from this.
        seed: same seed gives the same draw; a different seed gives a different one.
        n_per_doc: how many sentences to take from each document, for "by_document".

    Returns:
        The drawn items, renumbered from 1.

    Raises:
        ValueError: when strategy is not one of the three names.

    Example:
        >>> items = sample(pool, "balanced", N_PER_CLASS, seed=SEED)
    """
    n_labels = len(label_set(pool))
    n_total = n_per_class * n_labels

    if strategy == "balanced":
        return sample_pool(pool, n_per_class, seed)
    if strategy == "random":
        return sample_random(pool, n_total, seed)
    if strategy == "by_document":
        # Enough documents to reach the same total, rounded up so the draw is never
        # smaller than the other two strategies would have given.
        n_docs = max(1, -(-n_total // n_per_doc))
        return sample_by_document(pool, n_docs, n_per_doc, seed)

    raise ValueError(
        "strategy has to be balanced, random or by_document, and it says "
        + repr(strategy) + ". Fix the line above and run this cell again.")

**`sample_pool` — balanced across labels.** In: the pool and how many items you want per label. Out: up to that many of each. Step 1 sorts the pool into one bucket per label; step 2 takes up to `n_per_class` from each. A label with fewer than that gives all it has, which is why a rare class comes back short — that is data, not a bug. Step 3 gives the sample its own ids, 1, 2, 3 …, and keeps the id each item had in the pool as `source_id`. That second number is what lets `02b_add_samples.ipynb` add more items later without drawing one you already have.

In [ ]:
def sample_pool(pool: list[dict[str, str]],
                n_per_class: int,
                seed: int = 42) -> list[dict[str, str]]:
    """Pick up to n_per_class items for EACH label, chosen at random.

    Rare labels simply give fewer items - that is a property of the data.

    This is the BALANCED strategy: it forces the classes level so that per-class
    precision and recall are readable and the confusion matrix is not dominated by
    one huge class. The cost is that your sample no longer looks like the corpus.
    See sample_random and sample_by_document for the other two positions.

    Args:
        pool: the items to draw from, each {"id", "text", "label"}.
        n_per_class: how many to take per label.
        seed: same seed gives the same draw; a different seed gives a different one,
            so different groups can get different subsets.

    Returns:
        The drawn items, shuffled and renumbered from 1.

    Example:
        >>> items = sample_pool(pool, N_PER_CLASS, seed=SEED)
    """
    random_generator = random.Random(seed)

    # Step 1: sort every pool item into a bucket named after its label.
    items_by_label = {}
    for item in pool:
        label = item["label"]
        if label not in items_by_label:
            items_by_label[label] = []
        items_by_label[label].append(item)

    # Step 2: from each bucket, shuffle and keep up to n_per_class items.
    sampled = []
    for label in sorted(items_by_label):
        items_with_this_label = items_by_label[label]
        random_generator.shuffle(items_with_this_label)
        kept_items = items_with_this_label[:n_per_class]
        for item in kept_items:
            sampled.append(item)

    # Step 3: shuffle the whole set and renumber the ids from 1.
    random_generator.shuffle(sampled)
    sampled = reid(sampled)

    # Step 4: say what we drew, and warn if we took most of the pool. A sample is only
    # meaningful if there is a pool left over - both because a near-total sample is not
    # a sample, and because build_fewshot needs unused items for its examples.
    _report_draw(sampled, pool, "Sampled balanced by label")
    return sampled

**`sample_random` — the corpus as it is.** In: the pool and one total. Out: that many items, drawn without looking at the labels at all, so the draw keeps the pool's own imbalance. Realistic, and unkind to rare labels.

In [ ]:
def sample_random(pool: list[dict[str, str]],
                  n_total: int,
                  seed: int = 42) -> list[dict[str, str]]:
    """Draw n_total items at random, ignoring the labels entirely.

    The corpus as it actually is: every item equally likely, so each label turns up
    roughly as often as it does in the pool. That is the honest thing if you want to
    say something about the corpus - and the awkward thing if a label is rare, because
    a rare label will come back with one or two items, or none at all, and precision
    and recall on a class of one mean very little.

    Compare sample_pool, which forces the classes level instead.

    Args:
        pool: the items to draw from. It is copied, not shuffled in place.
        n_total: how many items to draw altogether.
        seed: same seed gives the same draw.

    Returns:
        The drawn items, renumbered from 1.

    Example:
        >>> items = sample_random(pool, 40, seed=SEED)
    """
    random_generator = random.Random(seed)

    # Copy before shuffling: shuffling the caller's pool in place would quietly change
    # the order of the list they are still holding.
    shuffled = list(pool)
    random_generator.shuffle(shuffled)
    sampled = reid(shuffled[:n_total])

    _report_draw(sampled, pool, "Sampled at random")
    return sampled

**`sample_by_document` — whole passages** (`cars50` and `raamove` only). In: the pool, how many documents, how many sentences from each. Out: that many sentences, drawn from that many documents. It is the longest of the four, and most of the length is the check it opens with: on a track whose items are loose sentences there are no documents to stratify by, so it stops and says so rather than inventing an answer.

In [ ]:
def sample_by_document(pool: list[dict[str, str]],
                       n_docs: int,
                       n_per_doc: int,
                       seed: int = 42) -> list[dict[str, str]]:
    """Pick whole documents first, then sentences inside them.

    Forty sentences drawn from forty abstracts and forty sentences drawn from three
    are both "forty sentences", and they support very different claims. This strategy
    makes that choice explicit: n_docs documents, n_per_doc sentences from each.

    Args:
        pool: the items to draw from. Every item must carry a "doc_id".
        n_docs: how many documents to draw.
        n_per_doc: how many sentences to take from each of them.
        seed: same seed gives the same draw.

    Returns:
        The drawn items, renumbered from 1.

    Raises:
        ValueError: when any item has no doc_id. Only cars50, cars50_step and
            raamove record which document a sentence came from.

    Example:
        >>> items = sample_by_document(pool, 10, 4, seed=SEED)
    """
    # Say it here, in terms of the track, rather than dying on a KeyError inside the
    # loop below - which would read as "the code is broken" rather than "this corpus
    # does not record which document a sentence came from".
    items_without_doc = 0
    for item in pool:
        if "doc_id" not in item:
            items_without_doc = items_without_doc + 1
    if items_without_doc > 0:
        raise ValueError(
            "sample_by_document needs to know which document each item came from, "
            "and " + str(items_without_doc) + " of these " + str(len(pool)) + " items "
            "do not carry a doc_id.\n"
            "Only the rhetorical-move tracks record that: cars50, cars50_step and "
            "raamove. On this track a sentence is not part of a passage in the data, "
            "so there are no documents to stratify by.\n"
            "Use sample_pool (balanced across labels) or sample_random instead, and "
            "say in PLAN.md which you chose.")

    random_generator = random.Random(seed)

    # Step 1: group the pool into documents.
    items_by_doc = {}
    for item in pool:
        doc_id = item["doc_id"]
        if doc_id not in items_by_doc:
            items_by_doc[doc_id] = []
        items_by_doc[doc_id].append(item)

    # Step 2: choose the documents. Sorted first, so the seed alone decides the draw -
    # dict order would otherwise depend on what order the pool happened to be built in.
    doc_ids = sorted(items_by_doc)
    random_generator.shuffle(doc_ids)
    chosen_docs = doc_ids[:n_docs]
    if len(chosen_docs) < n_docs:
        print("NOTE: you asked for", n_docs, "documents and the pool has only",
              len(chosen_docs), "- using all of them.")

    # Step 3: from each chosen document, take up to n_per_doc sentences.
    sampled = []
    for doc_id in chosen_docs:
        items_in_doc = list(items_by_doc[doc_id])
        random_generator.shuffle(items_in_doc)
        for item in items_in_doc[:n_per_doc]:
            sampled.append(item)

    random_generator.shuffle(sampled)
    sampled = reid(sampled)

    _report_draw(sampled, pool, "Sampled by document")
    print("        from", len(chosen_docs), "documents, up to", n_per_doc, "each.")
    # The labels were never controlled for, so say what that cost. A group that draws
    # by document and then reports per-class F1 needs to have seen this line.
    print("        Note the label counts above: this strategy balances DOCUMENTS,")
    print("        not labels, so a rare move stays rare.")
    return sampled

Run this to check the definitions above took effect. It prints the first line of the function the notebook will actually use, and the description of each argument.

In [ ]:
help(sample)

Now we draw the sample. The cell runs as written, using the balanced strategy — `sample_pool` is the one you ran on Day 4 Part A. The work is deciding whether that is the strategy your study wants, and being able to say why.

Every strategy prints its per-label counts. Run more than one and compare: the difference between them is the argument you have to make in your report.

All three are sized from `n_per_class` in `config.yaml` and take `SEED`, both already passed — so switching strategy changes how the items are chosen, not how many. A sample nobody can redraw is a sample nobody can check, and your report has to state the seed.

**If it warns that you took most of the pool**, you are almost certainly still pointed at `DEMO_POOL_PATH`.

In [ ]:
# ══ STEP 2 · Draw your sample ═════════════════════════════════════════════
# Draws the sample using whichever of the three strategies you name, and prints
# how many items landed under each label.
# Creates: sampled

# Balanced is the DEFAULT, not the recommendation. Change this one word to
# try another view of the corpus, and compare the counts each one prints.
#
# It is written as a choice rather than three lines you comment two of out,
# because with two live lines the SECOND one silently wins.
#
# by_document is cars50 · raamove ONLY. The other tracks have no documents
# to stratify by, and it will stop and tell you so.
STRATEGY = "balanced"     # "balanced" · "random" · "by_document"

sampled = sample(pool, STRATEGY, N_PER_CLASS, SEED)


Now we note which labels came out of that draw. `LABELS` is used again in step 4 below, and in notebook 03.

In [ ]:
LABELS = label_set(sampled)
print("labels:", LABELS)

### Now write down why you drew it that way

Not in the notebook — in `PLAN.md` §5, in a sentence. It belongs in your report's methodology section, and the Q&A may well ask you to defend it, so write it while the reason is still in your head.

The shape of the answer (another track's, so it is not yours to copy):

> We sampled by document, 10 documents × 4 sentences, because a move label describes a sentence's job *within* its abstract, and 40 loose sentences from 40 different papers would have thrown that away.

### Sanity-check what you drew

Three questions worth answering before you commit forty hand-annotations to it:

1. **Did you get what you asked for?** Compare the counts against your intention. A short class is your rare label hitting its ceiling.
2. **How many distinct texts is this?** On `cars50` and `raamove` — see above. Forty sentences from four documents is a much narrower claim.
3. **Is there pool left over?** `build_fewshot` in notebook 04 draws its examples from items you did *not* sample. If the sample is most of the pool, there is nothing uncontaminated left to draw from.

In [ ]:
# ══ STEP 3 · Check the draw ═══════════════════════════════════════════════
# Prints the size of the pool, the size of the sample, what is left over, and
# the count under each label. Nothing new is named — this is a check.

# Count how many sampled items carry each label, one item at a time.
counts = {}
for item in sampled:
    label = item["label"]
    if label not in counts:
        counts[label] = 0
    counts[label] = counts[label] + 1

print("pool:", len(pool))
print("sampled:", len(sampled))
print("left over for few-shot examples:", len(pool) - len(sampled))
print("per label:", counts)


### How many distinct texts is this? — `cars50` · `raamove` only

Now we count the documents your sentences came from. Skip this cell on the other tracks: their items are loose sentences with no document attached, so there is nothing to count.

Forty sentences from four introductions is a much narrower claim than forty from forty, and the difference belongs in your limitations section.

In [ ]:
# `doc_id` is one of the extra fields the move tracks carry. Collect the
# distinct ones, one item at a time.
documents = []
for item in sampled:
    doc_id = item.get("doc_id")
    if doc_id is not None and doc_id not in documents:
        documents.append(doc_id)

if documents:
    print("drawn from", len(documents), "distinct documents")
else:
    print("this track carries no doc_id — nothing to count here")

## Save the sample — it is what notebook 03 comes back to

The cell below writes the sample to a file. That is not ceremony even though the next cell uses it directly: notebook 03 runs **days later**, in a fresh Colab runtime, possibly for a different member of your group, and `sampled` will not exist in it. Adjudication needs these exact items back, to re-attach what the sheet does not carry.

Even with a fixed seed, save it — a seed reproduces a draw only as long as nobody edits the pool underneath it.

In [ ]:
save_json(sampled, SAMPLE_PATH, what="sampled items")

# It still carries the PUBLISHED label at this point. The sheet below
# deliberately does not copy that in — you annotate blind — but notebook 03
# uses it at the very end, to show you where your group disagreed with the
# corpus. That comparison is one of the more interesting things in your report.


## Step 4 — Create the annotation sheet

A real Google Sheet in your own Drive, one row per item, with blank `CoderA`, `CoderB`, `Final` and `Note` columns. All of you can have it open at once.

The published label is **deliberately not copied in**. Two people annotate independently, without seeing each other's column or the corpus's answer — that is what makes the agreement number mean anything. Decide who is CoderA and who is CoderB before you start, and do not look across.

**On `cars50` and `raamove`** the sheet gets one extra column, `Context`: the passage each sentence came from, with the sentence you are labelling marked `>>>`. Read it. A move is a rhetorical function *within* a passage, and if the sentence alone is too thin for the model to judge, it is just as thin for the two of you — and your labels are the answer key everything else gets measured against.

The first time you run this, Colab asks for permission to use your Google account. That is `gspread` authorising against your own Drive.

The sheet is created in the Drive of whoever runs the cell, so pass `share_with=MEMBERS` — the Google accounts you put in `config.yaml` — or your second coder will open the link and be told they need access. Pass `remember=SHEET_PATH` too, and the link is written to a file instead of living in this cell's output, where a runtime reset can lose it.

This is the Day 2 S5 step A call with those two sharing arguments added; it lives in `scripts/annotate.py`. The title is built for you out of your track, group and run, because you will have several of these sheets by the end of the week.

**Run this cell once.** Running it a second time makes a second sheet, and half your annotations end up in the one nobody reads back.

In [ ]:
# ══ STEP 4 · Create the sheet ═════════════════════════════════════════════
# Makes a Google Sheet with one row per sampled item and blank coder columns,
# shares it with your group, and writes the link to a file.

title = TRACK + " · " + GROUP + " · " + RUN + " annotation"

# Run this ONCE — a second run makes a second sheet.
url = create_annotation_sheet(title, sampled, LABELS,
                              share_with=MEMBERS,   # your group, from config.yaml
                              remember=SHEET_PATH)  # writes the link to a file
print(url)


---

## 🛑 This notebook is finished. Now go and annotate.

Open the sheet, and label every row. Rules of the exercise:

- **Each coder works in their OWN tab** — `CoderA`, `CoderB`. Different people, no discussion, and do not open a colleague's tab to see what they put. The sheet cannot stop you; the κ you report is only worth something if you do not.
- **Leave the `Final` tab alone** until everyone has finished and you have talked.
- **A third coder?** Duplicate an EMPTY tab (right-click ▸ *Duplicate*) and rename it `CoderC`. Copying a tab somebody has already filled in gives them that person's answers, and notebook 03 will tell you so in a warning you will not enjoy.
- **Use `Note`** when you hesitate. The item you were unsure about is the item you will want to quote in your error analysis, and you will not remember which one it was.
- Labels must be spelled exactly as `LABELS` prints them. `to_canonical` will tell you about typos, but it is quicker not to make them.

This is where the week's actual work happens, and it takes days rather than minutes.

> **Do not run this notebook again.** When you come back, open `03_annotate.ipynb` — it picks up from the file you just saved. Re-running the cells above would draw the sample *again*, and if anyone has touched `config.yaml` or the strategy line in the meantime, you would end up with a sample that no longer matches the sheet two people have been annotating. That is the kind of mistake you find out about in the Q&A.

**Next:** `03_annotate.ipynb`, once both columns are full.